# 04 — Baseline CNN Evaluation

Load the best checkpoint produced by notebook 02, run inference on the **test set**, and generate all diagnostic plots.

**Prerequisite:** the model package is installed in the active kernel and `artifacts/checkpoints/baseline_cnn_checkpoint.keras` exists (run notebook 02 first).

In [8]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import confusion_matrix, f1_score

from model_service.config import ModelServiceConfig
from model_service.preprocess.dataset_builder import build_pcam_datasets
from model_service.evaluation.plots import (
    plot_confusion_matrix,
    plot_roc,
    plot_pr_curve,
)

In [2]:
cfg = ModelServiceConfig()

checkpoint_path = cfg.paths.artifacts_models / "best_model.keras"
figures_dir = cfg.paths.artifacts_figures
figures_dir.mkdir(parents=True, exist_ok=True)

print("Loading checkpoint:", checkpoint_path)
model = tf.keras.models.load_model(checkpoint_path)
model.summary()

Loading checkpoint: /Users/shayan/code/sandinosaso/pathsight/artifacts/models/best_model.keras


2026-04-27 17:39:27.226505: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-04-27 17:39:27.226617: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-04-27 17:39:27.226631: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1777297167.227270 6072232 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1777297167.227538 6072232 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/shayan/.pyenv/versions/3.12.13/envs/pathsight/lib/python3.12/site-packages/keras/src/trainers/trainer.py:234: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(


Model: "baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 3, 3, 768)      │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,644,325 (307.63 MB)

 Trainable params: 26,362,817 (100.57 MB)

 Non-trainable params: 1,555,872 (5.94 MB)

 Optimizer params: 52,725,636 (201.13 MB)

In [9]:
def build_pcam_datasets_optimized(
    data_dir,
    download=False,
    max_train_samples=None,
    batch_size=32,
):
    # Load dataset
    (ds_train, ds_val, ds_test), ds_info = tfds.load(
        "patch_camelyon",
        split=["train", "validation", "test"],
        data_dir=data_dir,
        shuffle_files=True,
        as_supervised=True,
        with_info=True,
        download=download,
    )

    # Preprocessing function
    def prepare_data(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    # -------- TRAIN PIPELINE --------
    ds_train = ds_train.shuffle(10000)

    if max_train_samples:
        ds_train = ds_train.take(max_train_samples)

    ds_train = ds_train.map(
        prepare_data,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds_train = ds_train.batch(batch_size)

    ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

    # -------- VALIDATION PIPELINE --------
    ds_val = ds_val.map(
        prepare_data,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds_val = ds_val.batch(batch_size)
    ds_val = ds_val.prefetch(tf.data.AUTOTUNE)

    # -------- TEST PIPELINE --------
    ds_test = ds_test.map(
        prepare_data,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds_test = ds_test.batch(batch_size)
    ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

    return ds_train, ds_val, ds_test, ds_info

In [11]:
actual_data_path = "/Users/shayan/tensorflow_datasets"
# 2. Pass the data_dir (usually stored in cfg.paths.data_raw or similar)
# Check your ModelServiceConfig class to see the exact attribute name
_, _, test_ds, _ = build_pcam_datasets_optimized(
    data_dir=actual_data_path, 
    download=False
)

print("✅ Dataset built successfully!")

✅ Dataset built successfully!


In [15]:
results = model.evaluate(test_ds, return_dict=True)

print(f"📈 Full Test AUC: {results['auc']:.4f}")
print(f"🔍 Full Test Recall: {results['recall']:.4f}")
print(f"🎯 Full Test Accuracy: {results['accuracy']:.4f}")
print(f"⚠️ Full Test Loss: {results['loss']:.4f}")
print(f"🔍 Full Test Precision: {results['precision']:.4f}")

2026-04-27 17:58:56.597765: W tensorflow/core/framework/op_kernel.cc:1857] OP_REQUIRES failed at xla_ops.cc:591 : NOT_FOUND: could not find registered platform with id: 0x708f91b10


NotFoundError: Exception encountered when calling Conv2D.call().

[1mcould not find registered platform with id: 0x708f91b10 [Op:__inference__conv_xla_5699][0m

Arguments received by Conv2D.call():
  • inputs=tf.Tensor(shape=(32, 24, 24, 96), dtype=float32)

In [ ]:
# Collect ground-truth labels and predicted scores from the test set
y_true_list, y_score_list = [], []

for x_batch, y_batch in test_ds:
    preds = model.predict_on_batch(x_batch)   # shape (B, 1)
    y_score_list.append(preds.ravel())
    y_true_list.append(y_batch.numpy().ravel())

y_true  = np.concatenate(y_true_list).astype(np.int32)
y_score = np.concatenate(y_score_list).astype(np.float32)

print(f"Test samples: {len(y_true)}  |  Positive rate: {y_true.mean():.3f}")

🚚 Moving model weights from GPU to CPU...
🧪 Starting prediction...


2026-04-27 17:57:05.445987: W tensorflow/core/framework/op_kernel.cc:1857] OP_REQUIRES failed at xla_ops.cc:529 : INVALID_ARGUMENT: Trying to access resource convnext_tiny_stage_0_block_0_depthwise_conv/kernel/4 (defined @ /Users/shayan/.pyenv/versions/3.12.13/envs/pathsight/lib/python3.12/site-packages/keras/src/backend/tensorflow/core.py:42) located in device /job:localhost/replica:0/task:0/device:GPU:0 from device /job:localhost/replica:0/task:0/device:CPU:0
 Cf. https://www.tensorflow.org/xla/known_issues#tfvariable_on_a_different_device


InvalidArgumentError: Exception encountered when calling Conv2D.call().

[1mTrying to access resource convnext_tiny_stage_0_block_0_depthwise_conv/kernel/4 (defined @ /Users/shayan/.pyenv/versions/3.12.13/envs/pathsight/lib/python3.12/site-packages/keras/src/backend/tensorflow/core.py:42) located in device /job:localhost/replica:0/task:0/device:GPU:0 from device /job:localhost/replica:0/task:0/device:CPU:0
 Cf. https://www.tensorflow.org/xla/known_issues#tfvariable_on_a_different_device [Op:__inference__conv_xla_5652][0m

Arguments received by Conv2D.call():
  • inputs=tf.Tensor(shape=(32, 24, 24, 96), dtype=float32)

In [5]:
# Find the threshold that maximises F1 on the test set
thresholds = np.linspace(0.01, 0.99, 199)
f1_scores  = [f1_score(y_true, (y_score >= t).astype(int), zero_division=0) for t in thresholds]
best_threshold = float(thresholds[np.argmax(f1_scores)])
best_f1        = float(np.max(f1_scores))
print(f"Best F1 threshold: {best_threshold:.2f}  →  F1 = {best_f1:.4f}")

Best F1 threshold: 0.33  →  F1 = 0.8148


In [6]:
# ROC curve
plot_roc(
    y_true,
    y_score,
    out_path=figures_dir / "baseline_roc.png",
)
print("Saved:", figures_dir / "baseline_roc.png")

Saved: /Users/sandinosaso/code/sandinosaso/pathsight/artifacts/figures/baseline_roc.png


In [7]:
# Precision-Recall curve with best-F1 operating point
plot_pr_curve(
    y_true,
    y_score,
    best_f1_threshold=best_threshold,
    out_path=figures_dir / "baseline_pr_curve.png",
)
print("Saved:", figures_dir / "baseline_pr_curve.png")

Saved: /Users/sandinosaso/code/sandinosaso/pathsight/artifacts/figures/baseline_pr_curve.png


In [8]:
# Confusion matrix at the best-F1 threshold
y_pred = (y_score >= best_threshold).astype(np.int32)
cm     = confusion_matrix(y_true, y_pred)

plot_confusion_matrix(
    cm,
    labels=("Normal", "Metastatic"),
    out_path=figures_dir / "baseline_confusion_matrix.png",
)
print("Saved:", figures_dir / "baseline_confusion_matrix.png")
print(cm)

Saved: /Users/sandinosaso/code/sandinosaso/pathsight/artifacts/figures/baseline_confusion_matrix.png
[[12145  4246]
 [ 2198 14179]]
